### Libraries

In [ ]:
library(readxl)
library(openxlsx)
library(writexl)
library(pheatmap)
library(RColorBrewer)
library(stringr)
library(dplyr)
library(parallel)
library(doParallel)
library(foreach)

### CSI CALCULATION FUNCTION

In [ ]:
calculate_csi_matrix <- function(dist_obj, num_cores = NULL) {
    sample_names <- labels(dist_obj)
    n <- length(sample_names)
    
    if (is.null(num_cores)) {
        num_cores <- max(1, detectCores() - 1)
    }
    
    dist_matrix <- as.matrix(dist_obj)
    csi_matrix <- matrix(0, nrow = n, ncol = n)
    dimnames(csi_matrix) <- list(sample_names, sample_names)
    
    cl <- makeCluster(num_cores)
    registerDoParallel(cl)
    
    pairs <- expand.grid(i = 1:n, j = 1:n)
    pairs <- pairs[pairs$i < pairs$j, ]
    
    results <- foreach(idx = 1:nrow(pairs), .combine = "c") %dopar% {
        i <- pairs$i[idx]
        j <- pairs$j[idx]
        
        sample_a <- sample_names[i]
        sample_b <- sample_names[j]
        
        dist_ab <- dist_matrix[sample_a, sample_b]
        csi_count <- 0
        
        for (k in 1:n) {
            sample_x <- sample_names[k]
            if (sample_x != sample_a && sample_x != sample_b) {
                dist_ax <- dist_matrix[sample_a, sample_x]
                dist_bx <- dist_matrix[sample_b, sample_x]
                
                max_dist <- max(dist_ax, dist_bx, na.rm = TRUE)
                if (!is.na(max_dist) && !is.na(dist_ab) && max_dist < dist_ab) {
                    csi_count <- csi_count + 1
                }
            }
        }
        csi_count
    }
    stopCluster(cl)
    
    for (idx in 1:nrow(pairs)) {
        i <- pairs$i[idx]
        j <- pairs$j[idx]
        csi_matrix[i, j] <- results[idx]
        csi_matrix[j, i] <- results[idx]
    }
    return(csi_matrix)
}

### Paths & Metadata

In [ ]:
base_dir <- "/rprojectnb/cancergrp/brb/"
vtr_df <- read_excel(paste0(base_dir, "raw_data/VTR_INFORMATION.xlsx"))
vtr_codebook <- read_excel(paste0(base_dir, "raw_data/VTR_codebook.xlsx"), sheet = 1)

tables_folder <- paste0(base_dir, "intermediate_files/new_filtered_gsea_runs/gsea_tables/")
table_file <- paste0(tables_folder, "master_table.xlsx")
pvalue_file <- paste0(tables_folder, "pvalue_table.xlsx")

### Main Processing Loop

In [ ]:
for (use_ctrl in c(TRUE, FALSE)) {
    
    ctrl_status <- ifelse(use_ctrl, "with_ctrl", "without_ctrl")
    cat("\n======================================================\n")
    cat("PROCESSING DATA:", toupper(ctrl_status), "\n")
    cat("======================================================\n")
    
    # Define output folders
    out_dir_base <- paste0(base_dir, "intermediate_files/new_filtered_gsea_runs_corrected/heatmaps/NES/final_pipeline_", ctrl_status, "/")
    pearson_folder <- paste0(out_dir_base, "Pearson/")
    csi_folder <- paste0(out_dir_base, "CSI/")
    
    if (!dir.exists(pearson_folder)) dir.create(pearson_folder, recursive = TRUE)
    if (!dir.exists(csi_folder)) dir.create(csi_folder, recursive = TRUE)

    for (condition_id in c("stimulated", "unstimulated", "both")) {
        
        # Load master tables
        score_matrix_saved <- read.xlsx(table_file)
        p_value_matrix <- read.xlsx(pvalue_file)
                 
        if (!all(colnames(score_matrix_saved)[-1] == colnames(p_value_matrix)[-1])) {
            stop("The column names of the score and p-value matrices do not match.")
        }
        
        # Handle Control sample removal
        ctrl_sample <- "ctrl_unstimulated_Batch3.Batch4.Batch5.Batch6"
        if (!use_ctrl && ctrl_sample %in% colnames(score_matrix_saved)) {
            score_matrix_saved <- score_matrix_saved %>% select(-all_of(ctrl_sample))
            p_value_matrix <- p_value_matrix %>% select(-all_of(ctrl_sample))
        }

        # Separate Paths column from numeric data
        score_paths <- score_matrix_saved$Paths
        numeric_score_matrix <- as.matrix(score_matrix_saved[, -1])
        numeric_pvalue_matrix <- as.matrix(p_value_matrix[, -1])

        # FILTER 1: Set NES values to 0 where p-value is > 0.05
        numeric_score_matrix[numeric_pvalue_matrix > 0.05] <- 0

        # Temporarily combine to filter columns by condition
        full_filtered_df <- cbind(data.frame(Paths = score_paths), as.data.frame(numeric_score_matrix))

        if (condition_id == "both") {
            target_cols <- grep("_stimulated|_unstimulated", colnames(full_filtered_df), value = TRUE)
        } else {
            target_cols <- grep(paste0("_", condition_id), colnames(full_filtered_df), value = TRUE)
        }
        
        # Convert back to matrix
        score_matrix2 <- as.matrix(full_filtered_df[, target_cols])
        rownames(score_matrix2) <- full_filtered_df$Paths

        # Replace any remaining NAs with 0
        score_matrix2[is.na(score_matrix2)] <- 0

        # FILTER 2: Keep only rows (pathways) with non-zeros in >= 3 samples
        score_matrix2 <- score_matrix2[rowSums(score_matrix2 != 0) >= 3, ]
        
        rownames_score <- rownames(score_matrix2)
        score_matrix2 <- apply(score_matrix2, 2, as.numeric)
        rownames(score_matrix2) <- rownames_score

        # ---------------------------------------------------------------------
        # MATRICES CALCULATION
        # ---------------------------------------------------------------------
        # 1. Pearson Correlation
        cor_matrix <- cor(score_matrix2, method = "pearson")
        cor_dist <- as.dist(1 - cor_matrix) # Topological distance for Pearson
        
        # 2. CSI
        csi_matrix <- calculate_csi_matrix(cor_dist) # Feed correlation distance to CSI
        csi_matrix <- csi_matrix / (ncol(score_matrix2) - 2) # Normalize
        inverted_csi_matrix <- 1 - csi_matrix
        inverted_csi_dist <- as.dist(inverted_csi_matrix) # Topological distance for CSI

        # ---------------------------------------------------------------------
        # METADATA & ANNOTATIONS
        # ---------------------------------------------------------------------
        metadata_col <- data.frame(id = colnames(cor_matrix))
        metadata_col$batch_id <- str_split_fixed(metadata_col$id, '\\_', 4)[,3]
        metadata_col$condition <- str_split_fixed(metadata_col$id, '\\_', 4)[,2]
        metadata_col$vtr <- str_split_fixed(metadata_col$id, '\\_', 4)[,1]

        metadata_col$virus_type <- ifelse(metadata_col$vtr == "ctrl", "ctrl", 
                                          vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)])
        rownames(metadata_col) <- metadata_col$id
        
        n_virus_type <- length(unique(metadata_col$virus_type))
        n_batch_id <- length(unique(metadata_col$batch_id))
        n_condition <- length(unique(metadata_col$condition))
        
        ann_colors <- list(
            virus_type = setNames(colorRampPalette(brewer.pal(max(3, n_virus_type), "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
            batch_id = setNames(colorRampPalette(brewer.pal(max(3, n_batch_id), "Dark2"))(n_batch_id), unique(metadata_col$batch_id)),
            condition = setNames(colorRampPalette(brewer.pal(max(3, n_condition), "Paired"))(n_condition), unique(metadata_col$condition))
        )
        my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

        # ---------------------------------------------------------------------
        # PLOT AND EXPORT: PEARSON
        # ---------------------------------------------------------------------
        pearson_title <- paste("Pearson -", stringr::str_to_title(condition_id), "-", ifelse(use_ctrl, "With Ctrl", "No Ctrl"))
        pearson_file <- paste0(pearson_folder, "NES_Pearson_", condition_id)
        
        rg_p <- max(abs(cor_matrix), na.rm = TRUE); if (!is.finite(rg_p) || rg_p == 0) rg_p <- 1
        annot_cols <- if (condition_id == "both") c('virus_type', 'batch_id', 'condition') else c('virus_type', 'batch_id')

        out_p <- pheatmap(cor_matrix,
                          clustering_distance_rows = cor_dist,
                          clustering_distance_cols = cor_dist,
                          cluster_rows = TRUE, cluster_cols = TRUE,
                          breaks = seq(-rg_p, rg_p, length.out = 100),
                          clustering_method = "complete", color = my_colors,
                          show_rownames = FALSE, show_colnames = FALSE,
                          annotation_col = metadata_col[, annot_cols, drop = FALSE], annotation_colors = ann_colors,
                          border_color = 'black', main = pearson_title, fontsize = 8,
                          filename = paste0(pearson_file, "_heatmap.pdf"), width = 12, height = 10)

        reordered_pearson <- cor_matrix[out_p$tree_row[["order"]], out_p$tree_col[["order"]]]
        write_xlsx(data.frame(VTR = rownames(reordered_pearson), reordered_pearson, check.names = FALSE), paste0(pearson_file, ".xlsx"))
        
        # ---------------------------------------------------------------------
        # PLOT AND EXPORT: CSI
        # ---------------------------------------------------------------------
        csi_title <- paste("CSI -", stringr::str_to_title(condition_id), "-", ifelse(use_ctrl, "With Ctrl", "No Ctrl"))
        csi_file <- paste0(csi_folder, "NES_CSI_", condition_id)
        
        rg_c <- max(abs(csi_matrix), na.rm = TRUE); if (!is.finite(rg_c) || rg_c == 0) rg_c <- 1

        out_c <- pheatmap(csi_matrix,
                          clustering_distance_rows = inverted_csi_dist,
                          clustering_distance_cols = inverted_csi_dist,
                          cluster_rows = TRUE, cluster_cols = TRUE,
                          breaks = seq(0, rg_c, length.out = 100), # CSI scales 0 to 1
                          clustering_method = "complete", color = my_colors,
                          show_rownames = FALSE, show_colnames = FALSE,
                          annotation_col = metadata_col[, annot_cols, drop = FALSE], annotation_colors = ann_colors,
                          border_color = 'black', main = csi_title, fontsize = 8,
                          filename = paste0(csi_file, "_heatmap.pdf"), width = 12, height = 10)

        reordered_csi <- inverted_csi_matrix[out_c$tree_row[["order"]], out_c$tree_col[["order"]]]
        write_xlsx(data.frame(VTR = rownames(reordered_csi), reordered_csi, check.names = FALSE), paste0(csi_file, "_inverted.xlsx"))
        
        cat("Successfully processed & exported:", stringr::str_to_title(condition_id), "\n")
    }
}

### Fig. 2A: GSEA NES Heatmaps (Samples vs Pathways)

In [ ]:
library(dplyr)
library(readxl)
library(openxlsx)
library(writexl)
library(pheatmap)
library(RColorBrewer)
library(stringr)

In [ ]:
base_dir <- "/rprojectnb/cancergrp/brb/"
vtr_df <- read_excel(paste0(base_dir, "raw_data/VTR_INFORMATION.xlsx"))
vtr_codebook <- read_excel(paste0(base_dir, "raw_data/VTR_codebook.xlsx"), sheet = 1)

tables_folder <- paste0(base_dir, "intermediate_files/new_filtered_gsea_runs/gsea_tables/")
table_file <- paste0(tables_folder, "master_table.xlsx")
pvalue_file <- paste0(tables_folder, "pvalue_table.xlsx")

condition_id <- "unstimulated"

In [ ]:
# PROCESS UNSTIMULATED (WITH AND WITHOUT CONTROL)

for (use_ctrl in c(TRUE, FALSE)) {
    
    ctrl_status <- ifelse(use_ctrl, "with_ctrl", "without_ctrl")
    cat("\n======================================================\n")
    cat("PROCESSING UNSTIMULATED DATA:", toupper(ctrl_status), "\n")
    cat("======================================================\n")
    
    out_folder <- paste0(base_dir, "intermediate_files/new_filtered_gsea_runs_corrected/heatmaps/NES/pearson_direct_", ctrl_status, "/")
    if (!dir.exists(out_folder)) dir.create(out_folder, recursive = TRUE)

    # Load master tables
    score_matrix_saved <- read.xlsx(table_file)
    p_value_matrix <- read.xlsx(pvalue_file)
             
    # Handle Control sample removal (just like Fig 2A original code)
    ctrl_sample <- "ctrl_unstimulated_Batch3.Batch4.Batch5.Batch6"
    if (!use_ctrl && ctrl_sample %in% colnames(score_matrix_saved)) {
        score_matrix_saved <- score_matrix_saved %>% select(-all_of(ctrl_sample))
        p_value_matrix <- p_value_matrix %>% select(-all_of(ctrl_sample))
    }

    score_paths <- score_matrix_saved$Paths
    numeric_score_matrix <- as.matrix(score_matrix_saved[, -1])
    numeric_pvalue_matrix <- as.matrix(p_value_matrix[, -1])

    # Filter 1: Set NES values to 0 where p-value is > 0.05
    numeric_score_matrix[numeric_pvalue_matrix > 0.05] <- 0

    # Bind and extract ONLY unstimulated columns
    full_filtered_df <- cbind(data.frame(Paths = score_paths), as.data.frame(numeric_score_matrix))
    target_cols <- grep(paste0("_", condition_id), colnames(full_filtered_df), value = TRUE)
    
    score_matrix2 <- as.matrix(full_filtered_df[, target_cols])
    rownames(score_matrix2) <- full_filtered_df$Paths

    # Replace any remaining NAs with 0
    score_matrix2[is.na(score_matrix2)] <- 0

    # Filter 2: Keep rows with non-zeros in >= 3 samples
    score_matrix2 <- score_matrix2[rowSums(score_matrix2 != 0) >= 3, ]
    
    rownames_score <- rownames(score_matrix2)
    score_matrix2 <- apply(score_matrix2, 2, as.numeric)
    rownames(score_matrix2) <- rownames_score

    # ---------------------------------------------------------------------
    # DIRECT PEARSON CORRELATION
    # ---------------------------------------------------------------------
    cor_matrix <- cor(score_matrix2, method = "pearson")

    # ---------------------------------------------------------------------
    # METADATA & HEATMAP
    # ---------------------------------------------------------------------
    metadata_col <- data.frame(id = colnames(cor_matrix))
    metadata_col$batch_id <- str_split_fixed(metadata_col$id, '\\_', 4)[,3]
    metadata_col$vtr <- str_split_fixed(metadata_col$id, '\\_', 4)[,1]
    metadata_col$virus_type <- ifelse(metadata_col$vtr == "ctrl", "ctrl", 
                                      vtr_df$`Viral family`[match(metadata_col$vtr, vtr_df$code)])
    rownames(metadata_col) <- metadata_col$id
    
    n_virus_type <- length(unique(metadata_col$virus_type))
    n_batch_id <- length(unique(metadata_col$batch_id))
    
    ann_colors <- list(
        virus_type = setNames(colorRampPalette(brewer.pal(max(3, n_virus_type), "Set3"))(n_virus_type), unique(metadata_col$virus_type)),
        batch_id = setNames(colorRampPalette(brewer.pal(max(3, n_batch_id), "Dark2"))(n_batch_id), unique(metadata_col$batch_id))
    )
    my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(100)

    heatmap_title <- paste("Pearson (Direct) - Unstimulated -", ifelse(use_ctrl, "With Ctrl", "No Ctrl"))
    heatmap_file <- paste0(out_folder, "nes_pearson_direct_unstimulated")

    rg <- max(abs(cor_matrix), na.rm = TRUE)
    if (!is.finite(rg) || rg == 0) rg <- 1

    # Topological distance (1 - Pearson)
    cor_dist <- as.dist(1 - cor_matrix)
    vtr_annotations <- list(c('virus_type', 'batch_id'), c('virus_type'))
    
    for (heatmap_j in 1:length(vtr_annotations)) {
        annot_cols <- vtr_annotations[[heatmap_j]]

        out <- pheatmap(cor_matrix,
                        clustering_distance_rows = cor_dist,
                        clustering_distance_cols = cor_dist,
                        cluster_rows = TRUE, cluster_cols = TRUE,
                        breaks = seq(-rg, rg, length.out = 100),
                        clustering_method = "complete", color = my_colors,
                        show_rownames = FALSE, show_colnames = FALSE,
                        annotation_col = metadata_col[, annot_cols, drop = FALSE],
                        annotation_colors = ann_colors, border_color = 'black',
                        main = heatmap_title, fontsize = 8,
                        filename = paste0(heatmap_file, "_", heatmap_j, ".pdf"),
                        width = 12, height = 10)
    }

    # Extract & Save
    reordered_matrix <- cor_matrix[out$tree_row[["order"]], out$tree_col[["order"]]]
    write_xlsx(data.frame(VTR = rownames(reordered_matrix), reordered_matrix, check.names = FALSE), paste0(heatmap_file, ".xlsx"))
    
    cat("Successfully processed:", ctrl_status, "\n")
}